# Act 1 — KMS: the cryptographic root

Every other security service in this notebook either uses keys, holds keys, or proves access to keys. You cannot talk about cloud security without first knowing where the keys live, who controls them, and how the rest of AWS encrypts data without ever seeing those keys in plaintext.

**KMS** is the answer to all three. It is the managed service that creates and protects cryptographic keys, stores them inside FIPS-validated hardware modules, and provides the encrypt, decrypt, and generate-data-key APIs that every other AWS service uses underneath. This act covers the key types, **envelope encryption** (the pattern AWS uses everywhere to encrypt anything large), how access is granted through key policies and grants, rotation and deletion, multi-region keys, and the canonical KMS integrations in S3, EBS, and RDS.

## The six security layers at a glance

Security on AWS is layered. **KMS** is the cryptographic root — it creates and protects the keys that encrypt data across every other service. **Secrets Manager** (and its cheaper cousin SSM Parameter Store) holds credentials so applications never embed passwords or API keys. **Shield** absorbs DDoS at the network edge; **WAF** filters HTTP traffic at the application layer; **Firewall Manager** centralises both across an Organisation. **Cognito** handles user identity — who's logged in and what AWS resources they can reach. And a handful of detective services — **GuardDuty, Macie, Inspector, Detective** — watch for misuse after the fact.

## KMS — Key Management Service

KMS is the managed service that creates and protects cryptographic keys for AWS. Keys live inside FIPS 140-2 validated **hardware security modules** (HSMs) — they never leave KMS in plaintext, and every cryptographic operation happens inside the HSM. Almost every AWS service that encrypts data — S3, EBS, RDS, DynamoDB, Secrets Manager, SNS, SQS, Lambda — uses KMS underneath. Every API call is logged to CloudTrail, so encryption activity is fully audited.

### Key types by ownership

| Type | Created by | Controlled by | Cost | Rotation |
|---|---|---|---|---|
| **AWS managed key** | AWS, automatically | AWS | free | yearly, automatic |
| **Customer managed key (CMK)** | you | you | $1/month + API calls | optional (yearly or on-demand) |
| **AWS owned key** | AWS, shared across accounts | AWS (invisible) | free | AWS handles |

AWS managed keys are spun up automatically the first time you enable encryption on a service — `aws/s3`, `aws/ebs`, `aws/rds`, one per service per region. You can't see or modify their key policy. **Customer managed keys (CMKs)** are the ones you create yourself when you need full control: custom key policy, custom rotation schedule, cross-account sharing, the ability to disable or delete the key.

### Symmetric vs asymmetric

KMS keys come in two flavours.

- **Symmetric (AES-256)** — single key for encrypt and decrypt. The key never leaves KMS — to decrypt, you call KMS. Every AWS-service integration uses symmetric keys. This is the default and the one most architectures need.
- **Asymmetric (RSA or ECC)** — public/private key pair. KMS holds the private key; the public key is downloadable. Used for digital signatures, JWT verification, and cases where an external party encrypts data with your public key that only KMS can decrypt.

If the scenario is "encrypt data at rest in an AWS service," it's always a symmetric key. Asymmetric shows up for code signing, document signing, or cross-party encryption.

## Envelope Encryption

KMS has a hard limit of 4 KB per direct `Encrypt` call — and a per-region API rate limit that would collapse under bulk encryption load. The pattern AWS uses everywhere to work around this is **envelope encryption**.

The flow:

1. Application calls `GenerateDataKey(KeyId)`. KMS returns two things — a **plaintext data encryption key** (DEK), and the same DEK **encrypted under your KMS key**.
2. Application encrypts the actual data locally with the plaintext DEK (a fast symmetric cipher, no KMS round-trip per byte).
3. Application stores the encrypted data **together with the encrypted DEK**, and discards the plaintext DEK from memory.
4. To decrypt later: send the encrypted DEK to KMS, get the plaintext DEK back, decrypt the data locally.

The wins are stacked. Large payloads are encrypted at local CPU speed, not at KMS latency. The DEK never goes back to disk in plaintext. The encrypted DEK is safe to keep beside the data — useless without KMS access. And KMS only sees small keys, never the data itself.

Every AWS service integration with KMS (S3 SSE-KMS, EBS encryption, RDS at-rest, EFS, FSx, DynamoDB, every other one) implements envelope encryption under the hood. You almost never write the pattern yourself unless you're encrypting application-managed blobs.

## Key Policies, Grants, Cross-Account

KMS access control is **different** from most AWS services. For S3 or DynamoDB, IAM policies alone are enough to grant access. For KMS, the **key policy is the primary mechanism** — IAM policies only work if the key policy explicitly delegates to IAM.

A minimal key policy has two pieces.

- A statement enabling IAM — `Principal: arn:aws:iam::ACCOUNT:root, Action: kms:*` — which tells KMS to honour IAM policies for principals in your account. Without this, even an admin's IAM policy granting `kms:*` does nothing on this key.
- Statements granting specific principals specific actions — `kms:Decrypt`, `kms:Encrypt`, `kms:GenerateDataKey` — usually on `Resource: *` because the policy is attached to the key itself.

### Grants

**Grants** are a programmatic way to delegate specific permissions on a key without editing the policy. AWS services use them internally — when you encrypt an EBS volume, EBS creates a grant on the key permitting decrypt for that specific volume. Grants can be created and retired by code, which makes them the right tool for short-lived cross-account or service-to-service delegation.

### Cross-account

To use a KMS key from a different account, you need permission in two places:

1. The **key policy** in the key's account grants the foreign principal access.
2. The **IAM policy** in the foreign account grants the principal `kms:Decrypt` on the key ARN.

Both have to allow — either alone isn't enough. The cross-account pattern is the same as S3 bucket policy plus IAM.

## Rotation, Deletion, Multi-Region Keys

**Rotation.** Enable automatic rotation on a CMK and KMS generates new key material **once a year**. The key ID, ARN, and aliases stay the same — applications see no change. KMS keeps all previous key material versions, so data encrypted with old material is still decryptable transparently. **On-demand rotation** lets you trigger a rotation immediately, useful when you suspect compromise. AWS managed keys rotate automatically on the same yearly schedule and you can't disable it.

**Deletion.** KMS keys cannot be deleted immediately. Scheduling deletion starts a mandatory **waiting period of 7 to 30 days** (you choose), during which the key is disabled but recoverable. Cancel the deletion if you change your mind. Once the window closes, the key is gone — and **any data encrypted with it is permanently unrecoverable**. The safer move when you're unsure is to **disable** the key instead: it becomes unusable but no data is at risk, and you can re-enable later.

**Multi-Region Keys.** A set of KMS keys in different regions that share the same key material and the same key ID (prefixed `mrk-...`). Encrypt with the primary in `us-east-1`, decrypt with the replica in `eu-west-1` without re-encrypting. The use case is global applications, DynamoDB Global Tables, cross-region S3 replication of KMS-encrypted objects, and disaster-recovery copies. Each replica is independent for key policy purposes — you can grant different access in different regions.

## KMS in S3, EBS, and RDS

Three integrations that come up constantly.

### S3

Four server-side encryption modes:

| Mode | Keys | Notes |
|---|---|---|
| **SSE-S3** | AWS-managed, no KMS | default; cheapest; no per-request audit |
| **SSE-KMS** | KMS key (AWS managed or CMK) | every GET triggers a KMS `Decrypt`; logged in CloudTrail |
| **SSE-C** | customer-supplied per request | AWS encrypts/decrypts but never stores the key |
| **Client-side** | application encrypts before upload | AWS never sees plaintext at all |

The gotcha with SSE-KMS: **every S3 GET calls KMS to decrypt**. At high request rates that can hit the per-region KMS API rate limit (5,500 to 30,000 requests/sec depending on region). The fix is **S3 Bucket Keys** — S3 generates a short-lived bucket-level data key from KMS and uses it across many objects, dramatically cutting KMS API calls. Enable bucket keys whenever you use SSE-KMS at scale.

### EBS and RDS

**EBS encryption** is set at volume creation — you pick a key (`aws/ebs` or a CMK). Snapshots inherit the key automatically. The account-level **default encryption** setting forces every new volume to be encrypted, which is the right default. Copying a snapshot to another region re-encrypts with a key in the destination region.

**RDS encryption** is also set at instance creation and can't be added later. It covers storage, automated backups, read replicas, and snapshots. To encrypt an existing unencrypted RDS instance the workflow is: take a snapshot, copy the snapshot with encryption enabled, restore the encrypted copy as a new instance, repoint the application. Same flow appears for Aurora, DynamoDB, EFS, anywhere encryption has to be enabled at creation.

In [ ]:
import boto3

kms = boto3.client("kms", region_name="us-east-1")

# Customer managed key (symmetric, AES-256) with annual auto-rotation and a friendly alias
key = kms.create_key(Description="Production app encryption key",
                    KeyUsage="ENCRYPT_DECRYPT", KeySpec="SYMMETRIC_DEFAULT")
key_id = key["KeyMetadata"]["KeyId"]
kms.create_alias(AliasName="alias/prod-app-key", TargetKeyId=key_id)
kms.enable_key_rotation(KeyId=key_id)

# Envelope encryption — generate a DEK, encrypt the data locally with it, store the encrypted DEK
resp = kms.generate_data_key(KeyId="alias/prod-app-key", KeySpec="AES_256")
plaintext_dek, encrypted_dek = resp["Plaintext"], resp["CiphertextBlob"]
# encrypt_locally(data, plaintext_dek)  -- application-level AES
# persist (ciphertext, encrypted_dek); discard plaintext_dek from memory

# To decrypt later: send the encrypted DEK to KMS and get the plaintext DEK back
recovered = kms.decrypt(CiphertextBlob=encrypted_dek)["Plaintext"]
assert recovered == plaintext_dek

# Act 2 — Secrets without hard-coding

Encryption keeps data safe at rest. But applications still need credentials at runtime — database passwords, API keys, OAuth tokens, third-party service secrets. Hard-code them in config files and you have a leak waiting to happen. Embed them in environment variables and they show up in process listings and core dumps.

**Secrets Manager** is the answer for anything that needs automatic rotation — database credentials especially. **SSM Parameter Store** sits beside it for plain configuration that does not rotate. Many architectures use both.

## AWS Secrets Manager

Secrets Manager stores credentials — database passwords, API keys, OAuth tokens — and gives the application a single `GetSecretValue` call to retrieve them at runtime. Two things make it more than a glorified key-value store: **rotation** and **versioning**.

### Rotation

Secrets Manager can rotate secrets automatically on a schedule. The mechanism is a **rotation Lambda** that does four steps: create a new credential in the target service, update the secret value in Secrets Manager, test the new credential, retire the old one. Built-in rotation Lambdas exist for RDS (MySQL, PostgreSQL, Oracle, SQL Server, Aurora), Redshift, and DocumentDB; for anything else you write a custom rotation Lambda following the same four-step contract.

With rotation enabled, application code reads the current secret from Secrets Manager on every connection (or caches it briefly). When the secret rotates, the next read picks up the new value — no application restart, no manual deploy.

### Versioning

Secrets are versioned, with staging labels.

- **`AWSCURRENT`** — the live version that `GetSecretValue` returns by default
- **`AWSPENDING`** — the new version mid-rotation, being tested before promotion
- **`AWSPREVIOUS`** — the previous version, kept for rollback

During rotation there's always a valid current version — zero downtime. If the new credential fails its test, the rotation aborts and `AWSCURRENT` stays put.

### Replication and security

Secrets can be **replicated to other regions** with the primary and replicas kept in sync — used for multi-region apps, DR, and local-read latency. Secrets are always **encrypted at rest with KMS** — either the AWS managed `aws/secretsmanager` key or your CMK. Access is gated by **IAM policies** and **resource-based policies** on the secret itself, the same pattern as S3 bucket policies.

## Secrets Manager vs SSM Parameter Store

Both services hold configuration data; both can encrypt with KMS. The split is rotation, cost, and integration.

| | Secrets Manager | SSM Parameter Store |
|---|---|---|
| Cost | $0.40 / secret / month + API calls | free (Standard); $0.05 / parameter (Advanced) |
| Automatic rotation | **built in** with Lambda | none — manual only |
| Encryption | always KMS | optional KMS (SecureString type) |
| Max value size | 65 KB | 4 KB Standard, 8 KB Advanced |
| Cross-region replication | yes | no |
| RDS native integration | yes — managed RDS password lifecycle | no |

The rule: **rotation needed → Secrets Manager**. Database credentials, API keys with expiry, OAuth secrets — anywhere automatic rotation is part of the security story.

**Plain configuration → Parameter Store.** Feature flags, environment names, ARNs, non-secret config values. It's free at the Standard tier and integrates with everything (CloudFormation, ECS task definitions, Lambda environment variables can pull parameters at deploy time).

Many architectures use both — secrets in Secrets Manager, config in Parameter Store.

In [ ]:
import boto3, json

secrets = boto3.client("secretsmanager", region_name="us-east-1")
ssm     = boto3.client("ssm",            region_name="us-east-1")

# Secrets Manager — KMS-encrypted, rotated every 30 days by a managed RDS rotation Lambda
sec = secrets.create_secret(
    Name="prod/app/db-credentials",
    KmsKeyId="alias/prod-app-key",
    SecretString=json.dumps({"username": "appuser", "password": "initial!",
                              "host": "prod.cluster-abc.us-east-1.rds.amazonaws.com",
                              "port": 5432, "dbname": "appdb"}),
)
secrets.rotate_secret(
    SecretId=sec["ARN"],
    RotationLambdaARN="arn:aws:lambda:us-east-1:111122223333:function:SecretsManagerRDSPostgreSQLRotationSingleUser",
    RotationRules={"AutomaticallyAfterDays": 30},
    RotateImmediately=True,
)

# At runtime — single API call returns the current value
creds = json.loads(secrets.get_secret_value(SecretId="prod/app/db-credentials")["SecretString"])

# Parameter Store — non-rotating config, free, smaller payloads
ssm.put_parameter(Name="/prod/app/feature-flag", Value="true", Type="String", Overwrite=True)
ssm.put_parameter(Name="/prod/app/api-key", Value="sk-abc123", Type="SecureString",
                  KeyId="alias/prod-app-key", Overwrite=True)
flag = ssm.get_parameter(Name="/prod/app/api-key", WithDecryption=True)["Parameter"]["Value"]

# Act 3 — The perimeter: Shield, WAF, Firewall Manager

Encryption protects data at rest. Secrets protect credentials. The next layer is keeping bad traffic out before it reaches your application at all.

**Shield** absorbs DDoS at the network edge — Layer 3 and Layer 4 floods that would saturate your bandwidth, and with Advanced, Layer 7 floods that would exhaust your application's request budget. **WAF** filters HTTP traffic at the application layer — SQL injection, cross-site scripting, bad IPs, rate limits, geo-blocks. **Firewall Manager** centralises both of them across an entire Organisation so a new account inherits the baseline on day one.

## AWS Shield — DDoS Protection

Shield protects AWS resources from **distributed denial-of-service** attacks. Two tiers.

**Shield Standard** is automatic, free, and on for every AWS account. It defends **Layer 3 and Layer 4** — SYN floods, UDP reflection, volumetric attacks — for resources behind CloudFront, Route 53, Elastic Load Balancers, and EC2 Elastic IPs. Nothing to configure; AWS absorbs the routine DDoS traffic before it reaches your infrastructure.

**Shield Advanced** is a paid subscription at **$3,000 per organisation per month** (plus data transfer). It builds on Standard with several things worth memorising.

- **Layer 7 DDoS protection** — defends application-layer floods (HTTP GET/POST floods, slowloris) against CloudFront, ALB, API Gateway. Standard only covers L3/L4.
- **DDoS cost protection** — AWS credits the scaling charges (EC2, ELB, CloudFront, Route 53) you incur **because of** an active DDoS attack. Without this, a successful absorption can come with a six-figure bill.
- **24/7 access to the Shield Response Team (SRT)** — AWS experts who help mitigate live attacks, write custom WAF rules, and coordinate response.
- **Detailed attack metrics** and CloudWatch alarms — much richer visibility than Standard offers.
- **Automatic WAF rule creation** by the SRT during an incident.

The practical cue: "DDoS cost protection," "Layer 7 DDoS," "24/7 expert support," or "protect a high-value internet-facing application from DDoS" → Shield Advanced. Anything else → Standard is already covering you.

## AWS WAF — Web Application Firewall

WAF is a **Layer 7** filter for HTTP/HTTPS traffic — it inspects requests against rules you define and allows, blocks, counts, or CAPTCHA-challenges them before they reach your application. It attaches to:

- **CloudFront** (global — rules run at the edge)
- **Application Load Balancer** (regional)
- **API Gateway** (regional, REST APIs)
- **AppSync** (GraphQL APIs)
- **Cognito User Pools**

The top-level resource is a **Web ACL** — a container of rules with a default action (Allow or Block) for requests that match no rule. Rules are evaluated in priority order; the first matching rule's action wins.

### Rule types

| Rule | Matches |
|---|---|
| **IP set** | specific IPs or CIDR ranges |
| **Geo match** | by country |
| **Rate-based** | block IPs exceeding a per-5-minute request threshold |
| **String match** | URI, query string, headers, body — exact, prefix, regex |
| **SQL injection** | SQLi patterns in named fields |
| **XSS** | cross-site scripting patterns |
| **Size constraint** | oversized body / headers / URI |
| **Managed rule group** | pre-built rule sets from AWS or partners |

### Managed rule groups

AWS maintains a library of **managed rule groups** — pre-built, regularly updated, no tuning required. The big ones:

- **Core Rule Set** — OWASP Top 10
- **Known Bad Inputs** — patterns seen in real attacks
- **SQL Database / Linux / Windows** — engine-specific exploits
- **Bot Control** — scrapers, crawlers, automated traffic
- **Account Takeover Prevention** — credential-stuffing on login endpoints
- **Anonymous IP List** — Tor exit nodes, VPNs, proxies

The practical starting point for any new Web ACL is **Core Rule Set + Known Bad Inputs + a rate-based rule**.

### Rule actions

- **Allow** — pass the request through (terminal)
- **Block** — return HTTP 403 (terminal)
- **Count** — record the match and continue evaluating; used to dry-run a rule before flipping it to Block
- **CAPTCHA / Challenge** — present a CAPTCHA before allowing the request

WAF decisions log to Kinesis Firehose, CloudWatch Logs, or S3 — read them when debugging false positives or analysing attack traffic.

## AWS Firewall Manager

Firewall Manager centralises security policy across an **AWS Organisation** — WAF web ACLs, Shield Advanced subscriptions, security group baselines, Network Firewall policies, Route 53 Resolver DNS Firewall rules. Define a policy once in the management (or delegated administrator) account and Firewall Manager applies and enforces it across all member accounts.

The practical fit is two patterns. **Enforce a baseline**: every internet-facing ALB in every account must have the Core Rule Set WAF attached. **Onboard new accounts automatically**: a new account joining the Organisation inherits all policies on day one, without anyone having to remember.

The cue: "apply WAF or Shield across all accounts," "centralise security policy in an Organisation," "new accounts must automatically inherit" → Firewall Manager.

# Act 4 — User identity: Cognito

Everything so far has been about machine-to-machine security and traffic-level controls. But human users also need a way to sign in — to your app, to your API, eventually to AWS resources themselves.

**Cognito** is two distinct services under one name. **User Pools** answer "who are you?" — a managed user directory with sign-up, sign-in, MFA, social and SAML federation, and JWT issuance. **Identity Pools** answer "what AWS resources can you reach?" — they swap a token for temporary AWS credentials. They are usually paired but solve different problems.

## Cognito User Pools — Authentication

Cognito has two distinct products under one name. **User Pools** answer *who are you?* — a managed user directory with sign-up, sign-in, and identity tokens. **Identity Pools** answer *what AWS resources can you reach?* — they swap a token for temporary AWS credentials. They're usually paired, but solve different problems.

A **User Pool** is the authentication side. It handles:

- **Username/email + password** sign-up and sign-in with configurable password policies
- **Social identity providers** — Google, Facebook, Apple, Amazon — via federation
- **Enterprise federation** — SAML 2.0 and OIDC providers (Okta, Azure AD, ADFS, Auth0)
- **MFA** — SMS, TOTP authenticator apps
- **Email and phone verification**
- **Adaptive authentication** — risk-based MFA prompts on suspicious sign-ins (new device, impossible travel)
- **Hosted UI** — a pre-built, brandable sign-in page so applications don't have to ship a login form

On successful sign-in, the User Pool returns three **JWT tokens**: an **ID token** carrying the user's identity claims (name, email, custom attributes), an **access token** carrying authorisation scopes for your API, and a **refresh token** used to get new ID/access tokens without re-prompting the user.

### Integrations

- **API Gateway Cognito authoriser** — API Gateway validates the JWT against the User Pool with no Lambda authoriser code.
- **ALB authentication** — the load balancer redirects unauthenticated users to the Cognito Hosted UI, validates the returned token, and forwards user identity to the backend.

### Lambda triggers

User Pools fire Lambda functions at authentication lifecycle events — **pre sign-up** for validation or auto-confirm, **post confirmation** for welcome emails or DynamoDB writes, **pre authentication** for custom block logic, **post authentication** for logging, and **pre token generation** to inject custom claims into the JWT. The triggers are how you graft application-specific behaviour onto the managed auth flow.

**User Pool Groups** organise users into roles (admins, editors, viewers) and the group memberships flow into JWT claims, so authorisation logic in the application or in API Gateway can read them directly.

## Cognito Identity Pools — AWS Credentials

An **Identity Pool** does one job: take an identity token (from a Cognito User Pool, or directly from Google / Facebook / Apple / SAML / OIDC) and exchange it via STS `AssumeRoleWithWebIdentity` for **temporary AWS credentials**. The client then calls AWS services directly with those credentials — upload to S3, query DynamoDB, invoke a Lambda — without you running a backend that proxies the call.

### Identity sources

- Cognito **User Pools** (the most common pairing)
- **Social** — Google, Facebook, Apple, Amazon
- **SAML 2.0** identity providers
- **OpenID Connect** providers
- **Unauthenticated guests** — limited credentials for anonymous users (e.g. a read-only role for a public app)

### Roles

Each Identity Pool is wired to **two IAM roles** — an **authenticated** role used after a successful identity exchange, and an **unauthenticated** role used for guest access (if enabled). You can also do **role-based access control**: map users to different IAM roles based on their User Pool group or specific token claims, so admins get one role and regular users get another.

### User Pool + Identity Pool together

The canonical mobile-and-SPA pattern: the User Pool authenticates the user and issues JWTs; the Identity Pool exchanges the JWT for temporary AWS credentials scoped to a role; the application uses those credentials to call S3, DynamoDB, or AWS-native APIs directly from the client. The backend never sees the password and rarely sees the data path — the JWT and the temporary credentials carry everything.

The short distinction: **User Pool = identity tokens for your API. Identity Pool = AWS credentials for AWS APIs.**

In [ ]:
import boto3, json

wafv2  = boto3.client("wafv2",          region_name="us-east-1")
idp    = boto3.client("cognito-idp",    region_name="us-east-1")   # User Pools
fed    = boto3.client("cognito-identity", region_name="us-east-1")  # Identity Pools

# Web ACL for an ALB — block bad IPs, rate-limit, then run the AWS Core Rule Set
ip_set = wafv2.create_ip_set(Name="bad-ips", Scope="REGIONAL", IPAddressVersion="IPV4",
                              Addresses=["203.0.113.0/24"])
wafv2.create_web_acl(
    Name="app-web-acl", Scope="REGIONAL",
    DefaultAction={"Allow": {}},
    Rules=[
        {"Name": "BlockBadIPs", "Priority": 1, "Action": {"Block": {}},
         "Statement": {"IPSetReferenceStatement": {"ARN": ip_set["Summary"]["ARN"]}},
         "VisibilityConfig": {"SampledRequestsEnabled": True, "CloudWatchMetricsEnabled": True,
                              "MetricName": "BlockBadIPs"}},
        {"Name": "RateLimit", "Priority": 2, "Action": {"Block": {}},
         "Statement": {"RateBasedStatement": {"Limit": 2000, "AggregateKeyType": "IP"}},
         "VisibilityConfig": {"SampledRequestsEnabled": True, "CloudWatchMetricsEnabled": True,
                              "MetricName": "RateLimit"}},
        {"Name": "CoreRuleSet", "Priority": 3, "OverrideAction": {"None": {}},
         "Statement": {"ManagedRuleGroupStatement": {"VendorName": "AWS",
                                                       "Name": "AWSManagedRulesCommonRuleSet"}},
         "VisibilityConfig": {"SampledRequestsEnabled": True, "CloudWatchMetricsEnabled": True,
                              "MetricName": "CoreRuleSet"}},
    ],
    VisibilityConfig={"SampledRequestsEnabled": True, "CloudWatchMetricsEnabled": True,
                      "MetricName": "AppWebACL"},
)

# Cognito User Pool with strong password policy and email-based sign-in
pool = idp.create_user_pool(
    PoolName="MyAppUsers",
    Policies={"PasswordPolicy": {"MinimumLength": 12, "RequireUppercase": True,
                                  "RequireLowercase": True, "RequireNumbers": True,
                                  "RequireSymbols": True}},
    MfaConfiguration="OPTIONAL",
    AutoVerifiedAttributes=["email"], UsernameAttributes=["email"],
)
user_pool_id = pool["UserPool"]["Id"]

# App Client for a browser SPA — public client, OAuth authorization-code flow
client = idp.create_user_pool_client(
    UserPoolId=user_pool_id, ClientName="web-app", GenerateSecret=False,
    AllowedOAuthFlows=["code"], AllowedOAuthScopes=["openid", "email", "profile"],
    CallbackURLs=["https://myapp.example.com/callback"],
    AllowedOAuthFlowsUserPoolClient=True, SupportedIdentityProviders=["COGNITO", "Google"],
)

# Identity Pool wired to the User Pool — exchanges JWTs for temporary AWS credentials
fed.create_identity_pool(
    IdentityPoolName="MyAppIdentityPool",
    AllowUnauthenticatedIdentities=False,
    CognitoIdentityProviders=[{
        "ProviderName": f"cognito-idp.us-east-1.amazonaws.com/{user_pool_id}",
        "ClientId": client["UserPoolClient"]["ClientId"], "ServerSideTokenCheck": True,
    }],
)

# Act 5 — Detective controls and picking the right tool

Preventive controls — encryption, secrets, perimeter, identity — block known threats. But you cannot anticipate everything that will go wrong. **Detective controls** watch for unusual behaviour after the fact and surface findings for investigation.

This act covers the four detective services — **GuardDuty**, **Macie**, **Inspector**, **Detective** — and **Security Hub**, which aggregates them. It closes with a cue-to-service consolidation across the entire notebook.

## Detective Services — GuardDuty, Macie, Inspector, Detective

Four services that *detect* problems after the fact — none of them block traffic; all of them surface findings for you (or Security Hub) to act on.

**GuardDuty.** Continuous threat detection across CloudTrail logs, VPC Flow Logs, DNS query logs, and (optionally) S3 data events, EKS audit logs, and EBS volumes for malware. Machine-learning models flag anomalies — unusual API calls, communication with known-bad IPs, crypto-mining patterns, credential exfiltration. Enable once at the Organisation level and it covers every account. Findings come out as EventBridge events so you can route them to SNS, Lambda, or Security Hub.

**Macie.** Sensitive data discovery in **S3**. Macie scans buckets for personally identifiable information (PII), financial data, credentials, healthcare data, intellectual property — using both regex patterns and machine learning. The output is a per-bucket inventory of what sensitive data lives where, and findings when something exposes itself (public bucket containing SSNs, unencrypted bucket with credit-card data). Macie is the right answer for compliance audits over S3 data lakes.

**Inspector.** Continuous vulnerability assessment for **EC2 instances**, **container images in ECR**, and **Lambda functions**. Scans for known CVEs in OS packages and application libraries, and for unintended network exposure on EC2. Inspector replaced the older host-based scanner — it now runs automatically on resources it discovers, with findings flowing into Security Hub.

**Detective.** Incident investigation. Detective pulls in CloudTrail, VPC Flow Logs, and GuardDuty findings and stitches them into a graph linking principals, resources, IP addresses, and time. When GuardDuty fires a finding, Detective is where you go to answer "what else did that principal touch, when, and from where?" without writing Athena queries by hand.

**Security Hub** sits over all four — it aggregates findings from GuardDuty, Macie, Inspector, Detective, IAM Access Analyzer, Config, and third-party tools into one place with a CIS / PCI / Foundational Security Best Practices compliance view. It's the standard control plane for a security team operating across many accounts.

The cue map: *unusual API behaviour or known-bad-IP traffic* → GuardDuty. *S3 holds sensitive data* → Macie. *EC2 / ECR / Lambda vulnerability scanning* → Inspector. *Investigate after a finding* → Detective. *Aggregate findings across all of these and check compliance* → Security Hub.

## Picking the Right Tool

The whole notebook as a cue-to-service sheet.

| Cue | Service |
|---|---|
| Encrypt anything at rest in AWS | **KMS** (CMK for control; AWS managed key otherwise) |
| Encrypt data > 4 KB or at high throughput | **envelope encryption** (`GenerateDataKey`) |
| Same encrypted data usable in multiple regions | **Multi-Region KMS keys** |
| Reduce KMS API calls from S3 GETs | **S3 Bucket Keys** |
| Store credentials with automatic rotation | **Secrets Manager** |
| Store config, feature flags, non-rotating values | **SSM Parameter Store** |
| Free DDoS protection for L3/L4 | **Shield Standard** (always on) |
| L7 DDoS, DDoS cost protection, 24/7 SRT | **Shield Advanced** |
| Filter HTTP traffic — SQLi, XSS, rate limit, geo block | **WAF** (Web ACL on CloudFront/ALB/API Gateway) |
| Pre-built attack-pattern rules | **WAF Managed Rule Groups** (Core Rule Set, Known Bad Inputs, Bot Control) |
| Apply WAF / Shield / SG policies across all Organisation accounts | **Firewall Manager** |
| User sign-up / sign-in / social / SAML federation | **Cognito User Pools** |
| Temporary AWS credentials for client apps | **Cognito Identity Pools** |
| Detect unusual API or network activity | **GuardDuty** |
| Find sensitive data in S3 (PII, PHI, secrets) | **Macie** |
| Vulnerability scan EC2, ECR images, Lambda | **Inspector** |
| Investigate a security finding end-to-end | **Detective** |
| Aggregate findings + compliance dashboard | **Security Hub** |

A defensible production stack pulls from most of these: KMS encrypting everything at rest, Secrets Manager holding the database credentials, Shield Standard + WAF (with managed rule groups) fronted by CloudFront/ALB, Firewall Manager enforcing it across accounts, Cognito handling user identity, and GuardDuty + Inspector + Macie watching the rest with findings flowing into Security Hub. None of them is doing the whole job — each owns a layer.